# PDB Workstation · Re-docking con GNINA 1.3

Cuaderno complementario de la **PDB Drug Discovery Workstation** (CICATA-IPN).

Evalúa la calidad de una estructura del PDB como **sistema de docking** mediante el
re-docking de su ligando co-cristalizado, siguiendo la heurística de
**Domínguez-Ramírez et al. 2025** (*Quality over quantity*):

- Un **CNN score ≥ 0.9** indica un receptor de alta calidad para docking.
- El **RMSD** respecto a la pose cristalográfica es un dato de apoyo (no un filtro rígido).

Copia los valores de **CNN score** y **RMSD** que arroja este cuaderno a la pestaña
*Selección de Receptor* de la Workstation.

---
**Requisito:** activa la GPU en `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.


## 1. Comprobar GPU


In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode != 0:
    print('⚠️  No hay GPU activa. Ve a Entorno de ejecución → Cambiar tipo de entorno → T4 GPU y reinicia.')
else:
    print(r.stdout.split('\n')[8] if len(r.stdout.split('\n'))>8 else r.stdout)
    print('✓ GPU lista')


## 2. Instalar GNINA 1.3 y dependencias

Descarga el binario precompilado (no requiere compilar, ~1 min) e instala Open Babel para
la preparación de archivos.


In [ ]:
import os, subprocess, urllib.request

# Open Babel para conversión/preparación de ligandos
print('Instalando Open Babel...')
subprocess.run(['apt-get','-qq','install','-y','openbabel'], capture_output=True)

# Binario precompilado de GNINA 1.3 (incluye casi todas las dependencias)
GNINA_URL = 'https://github.com/gnina/gnina/releases/download/v1.3/gnina'
if not os.path.exists('/usr/local/bin/gnina'):
    print('Descargando GNINA 1.3 (~300 MB)...')
    urllib.request.urlretrieve(GNINA_URL, '/usr/local/bin/gnina')
    os.chmod('/usr/local/bin/gnina', 0o755)

# Verificar
v = subprocess.run(['gnina','--version'], capture_output=True, text=True)
print(v.stdout[:200] if v.stdout else v.stderr[:200])
print('✓ GNINA instalado')


## 3. Funciones de preparación

Descarga la estructura desde RCSB, separa la proteína (solo ATOM) del ligando co-cristalizado
y filtra solventes/iones comunes.


In [ ]:
import urllib.request, os

SOLVENTS = {'HOH','WAT','DOD','SO4','PO4','EDO','MPD','GOL','PEG','DMS','ACT','CIT',
            'FMT','MES','EPE','BME','DTT','TLA','ACE','EOH','IMD','TRS','DIO','IPA',
            'NAG','MAN','GLC','BMA','FUC','GAL','ZN','MG','NA','CL','K','CA','MN'}

def fetch_pdb(pdb_id):
    pdb_id = pdb_id.strip().upper()
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    txt = urllib.request.urlopen(url).read().decode()
    with open(f'{pdb_id}.pdb','w') as f: f.write(txt)
    return txt

def list_ligands(txt):
    ligs = {}
    for l in txt.split('\n'):
        if l[:6].strip()=='HETATM':
            code = l[17:20].strip()
            if code not in SOLVENTS:
                ligs[code] = ligs.get(code,0)+1
    return ligs

def split_structure(txt, pdb_id, lig_code):
    prot = [l for l in txt.split('\n') if l[:6].strip() in ('ATOM','TER')]
    lig  = [l for l in txt.split('\n') if l[:6].strip()=='HETATM' and l[17:20].strip()==lig_code]
    with open(f'{pdb_id}_protein.pdb','w') as f: f.write('\n'.join(prot)+'\nEND\n')
    with open(f'{pdb_id}_{lig_code}.pdb','w') as f: f.write('\n'.join(lig)+'\nEND\n')
    return f'{pdb_id}_protein.pdb', f'{pdb_id}_{lig_code}.pdb', len(lig)

print('✓ Funciones cargadas')


## 4. Función de re-docking

Ejecuta GNINA usando `--autobox_ligand` (la caja se define a partir del propio co-cristal)
y parsea el **CNN score**, **CNN affinity**, **afinidad** y **RMSD** respecto a la pose original.


In [ ]:
import subprocess, re, gzip, math

def parse_pdb_coords(path, hetatm=True):
    pts=[]
    for l in open(path):
        tag=l[:6].strip()
        if (tag=='HETATM' and hetatm) or (tag=='ATOM' and not hetatm):
            try: pts.append((float(l[30:38]),float(l[38:46]),float(l[46:54]),l[12:16].strip(),l[76:78].strip()))
            except: pass
    return pts

def heavy_rmsd_from_sdf(sdf_gz, ref_pdb):
    '''RMSD aproximado entre la mejor pose (SDF) y el co-cristal (PDB), átomos pesados.'''
    ref = [(x,y,z) for x,y,z,name,el in parse_pdb_coords(ref_pdb,True) if el!='H' and not name.startswith('H')]
    # leer primer modelo del SDF
    with gzip.open(sdf_gz,'rt') as f: lines=f.read().split('\n')
    try:
        natoms=int(lines[3][:3])
    except: return None
    pose=[]
    for i in range(4,4+natoms):
        p=lines[i].split()
        if len(p)>=4 and p[3]!='H': pose.append((float(p[0]),float(p[1]),float(p[2])))
    if not ref or not pose or len(ref)!=len(pose):
        return None  # conteos distintos → RMSD no comparable directamente
    s=sum((a[0]-b[0])**2+(a[1]-b[1])**2+(a[2]-b[2])**2 for a,b in zip(ref,pose))
    return math.sqrt(s/len(ref))

def redock(pdb_id, prot, lig, seed=0):
    out=f'{pdb_id}_redock.sdf.gz'
    cmd=['gnina','-r',prot,'-l',lig,'--autobox_ligand',lig,
         '--cnn_scoring','rescore','--seed',str(seed),'-o',out,'--no_gpu' if False else '']
    cmd=[c for c in cmd if c]
    res=subprocess.run(cmd, capture_output=True, text=True)
    log=res.stdout+res.stderr
    # parsear primera pose de la tabla de resultados
    cnn=cnnaff=aff=None
    for line in log.split('\n'):
        m=re.match(r'\s*1\s+(-?[0-9.]+)\s+([0-9.]+)\s+([0-9.]+)', line)
        if m:
            aff=float(m.group(1)); cnn=float(m.group(2)); cnnaff=float(m.group(3)); break
    rmsd=None
    try: rmsd=heavy_rmsd_from_sdf(out, lig)
    except: pass
    return {'pdb':pdb_id,'lig':lig.split('_')[-1].replace('.pdb',''),
            'affinity':aff,'cnn_score':cnn,'cnn_affinity':cnnaff,'rmsd':rmsd,'log':log}

print('✓ Función de re-docking lista')


## 5A. Opción A — Pegar PDB IDs

Escribe los códigos PDB separados por coma (ej. los que seleccionaste en la Workstation).
El cuaderno descarga cada estructura, detecta su ligando principal y re-dockea.


In [ ]:
#@title Ingresar PDB IDs { display-mode: 'form' }
PDB_IDS = '4ZUD, 6H5W'  #@param {type:'string'}
LIGANDO_FORZADO = ''     #@param {type:'string'}
#@markdown Deja *LIGANDO_FORZADO* vacío para autodetectar el ligando más grande.

ids=[x.strip().upper() for x in PDB_IDS.split(',') if x.strip()]
resultados=[]
for pid in ids:
    print(f'\n=== {pid} ===')
    try:
        txt=fetch_pdb(pid)
    except Exception as e:
        print(f'  ✗ No se pudo descargar: {e}'); continue
    ligs=list_ligands(txt)
    if not ligs:
        print('  ✗ Sin ligandos no-solvente detectados'); continue
    lig_code = LIGANDO_FORZADO.strip().upper() if LIGANDO_FORZADO.strip() else max(ligs,key=ligs.get)
    print(f'  Ligando: {lig_code} ({ligs.get(lig_code,0)} átomos) | otros: {list(ligs)}')
    prot,ligf,n=split_structure(txt,pid,lig_code)
    print(f'  Re-dockeando...')
    r=redock(pid,prot,ligf)
    resultados.append(r)
    cnn=f"{r['cnn_score']:.3f}" if r['cnn_score'] is not None else 'n/d'
    rmsd=f"{r['rmsd']:.2f}" if r['rmsd'] is not None else 'n/d'
    print(f"  → CNN score: {cnn} | RMSD: {rmsd} Å")
print('\n✓ Listo')


## 5B. Opción B — Subir archivos

Si prefieres usar archivos propios (por ejemplo el ZIP descargado de la Workstation, o tus
propios `.pdb`), ejecútalo aquí. Sube un **PDB completo** por estructura; el cuaderno separa
proteína y ligando automáticamente.


In [ ]:
from google.colab import files
import os

print('Sube uno o más archivos .pdb (estructura completa con co-cristal):')
up=files.upload()
resultados_b=[]
for fname in up:
    if not fname.lower().endswith('.pdb'): 
        print(f'  (omitido {fname})'); continue
    pid=os.path.splitext(fname)[0].upper()
    txt=open(fname).read()
    ligs=list_ligands(txt)
    if not ligs:
        print(f'  ✗ {fname}: sin ligandos'); continue
    lig_code=max(ligs,key=ligs.get)
    print(f'\n=== {pid} === ligando {lig_code}')
    prot,ligf,n=split_structure(txt,pid,lig_code)
    r=redock(pid,prot,ligf)
    resultados_b.append(r)
    cnn=f"{r['cnn_score']:.3f}" if r['cnn_score'] is not None else 'n/d'
    rmsd=f"{r['rmsd']:.2f}" if r['rmsd'] is not None else 'n/d'
    print(f"  → CNN score: {cnn} | RMSD: {rmsd} Å")
print('\n✓ Listo')


## 6. Tabla resumen

Consolida los resultados de ambas opciones. **Copia CNN score y RMSD** a la pestaña
*Selección de Receptor* de la Workstation. Recuerda: CNN ≥ 0.9 = alta calidad.


In [ ]:
import pandas as pd
todos = (resultados if 'resultados' in dir() else []) + (resultados_b if 'resultados_b' in dir() else [])
if not todos:
    print('No hay resultados todavía. Ejecuta la opción A o B.')
else:
    df=pd.DataFrame([{
        'PDB':r['pdb'],'Ligando':r['lig'],
        'CNN score':round(r['cnn_score'],3) if r['cnn_score'] is not None else None,
        'CNN affinity':round(r['cnn_affinity'],3) if r['cnn_affinity'] is not None else None,
        'Afinidad (kcal/mol)':round(r['affinity'],2) if r['affinity'] is not None else None,
        'RMSD (Å)':round(r['rmsd'],2) if r['rmsd'] is not None else None,
        'Calidad':'Alta (≥0.9)' if (r['cnn_score'] or 0)>=0.9 else ('Media' if (r['cnn_score'] or 0)>=0.5 else 'Baja'),
    } for r in todos])
    df=df.sort_values('CNN score',ascending=False,na_position='last').reset_index(drop=True)
    display(df)
    df.to_csv('gnina_redocking_resultados.csv',index=False)
    print('\nGuardado como gnina_redocking_resultados.csv')


---
### Notas metodológicas

- El **CNN score** evalúa la calidad de la pose con la red neuronal de GNINA; es más confiable
  que la afinidad sola para juzgar si el receptor reproduce el modo de unión conocido.
- El **RMSD** que calcula este cuaderno es una estimación de átomos pesados que asume el mismo
  orden atómico entre la pose y el co-cristal; cuando los conteos difieren, se reporta `n/d`.
  Para un RMSD riguroso usa **DockRMSD** o **obrms** (mapeo simétrico).
- Las estructuras con **CNN score bajo** suelen tener el sitio ocluido o residuos faltantes,
  algo que la calidad cristalográfica global (R-free, RSCC) no siempre revela.

Cita: McNutt et al., *GNINA 1.3*, J. Cheminformatics 2025; Domínguez-Ramírez et al.,
*Quality over quantity*, Front. Bioinform. 2025.
